# BiChannelRNN -- `hard` gate -- suthana 3-arm nested CV

Runs the **hard** variant of `BiChannelRNN` through the 3x3x3 nested
cross-validation on the suthana 3-arm bandit dataset.

The four gate variants (open each in its own Colab tab to run in parallel):
- `additive`  -- no gate, the two channels are simply summed
- `learnable` -- soft gate in (0,1), a free MLP on raw inputs
- `surprise`  -- soft gate driven by the explicit reward-prediction error
- `hard`      -- like `learnable` but the gate is binarised to {0,1}
  (straight-through estimator: a hard switch between the two channels)

**Runtime: TPU** -- Runtime > Change runtime type > TPU if not already set.
Results go to Google Drive, so a Colab disconnect does not lose progress:
reconnect and re-run the **Run CV** cell.

In [ ]:
# clone repo + install deps (jax is preinstalled and TPU-configured on Colab)
!git clone https://github.com/YifeiCAO/CogModelingRNNsTutorial.git
%cd CogModelingRNNsTutorial
!pip install -q dm-haiku optax

In [ ]:
import jax
print("JAX devices :", jax.devices())
print("Backend     :", jax.default_backend())
if jax.default_backend() != "tpu":
    print("WARNING: not on TPU -> Runtime > Change runtime type > TPU, then re-run.")

In [ ]:
# mount Google Drive so results survive a Colab disconnect (enables resume)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, time, pickle
import numpy as np
sys.path.insert(0, '/content/CogModelingRNNsTutorial')

# ================= EDIT HERE =================
GATE_MODE = "hard"            # additive | learnable | surprise | hard
DATASET   = "suthana"             # 3-arm bandit (n_actions = 3)
N_OUTER, N_INNER, N_SEEDS = 3, 3, 3
N_STEPS_MAX, PATIENCE     = 50000, 200
# quick test instead of the full run:  set N_SEEDS = 1 ; N_STEPS_MAX = 5000
# =============================================

# persistent output on Drive: one folder per gate mode (notebooks won't collide)
OUTPUT = f"/content/drive/MyDrive/bichannel_cv/{GATE_MODE}"
os.makedirs(OUTPUT, exist_ok=True)

import cv_runner
cv_runner.OUTPUT_DIR = OUTPUT     # send outer pkl + inner-search checkpoints to Drive
from cv_runner import run_outer_cv, make_bichannel_builder, HYBRID_GRID

d = np.load(f"colab_data/{DATASET}.npz")
xs, ys = d["xs"], d["ys"]
subject_ids = d["subject_ids"].astype(int) if "subject_ids" in d.files else None
print(f"{DATASET}: xs={xs.shape}  ys={ys.shape}  gate_mode={GATE_MODE}")
print(f"output -> {OUTPUT}")

t0 = time.time()
res = run_outer_cv(
    model_builder=make_bichannel_builder(GATE_MODE, n_actions=3, use_twostep=False),
    xs=xs, ys=ys,
    dataset_name=f"{DATASET}_bichannel_{GATE_MODE}",
    param_grid=HYBRID_GRID,
    group_ids=subject_ids,
    n_outer=N_OUTER, n_inner=N_INNER, n_seeds=N_SEEDS,
    n_steps_max=N_STEPS_MAX, early_stop_patience=PATIENCE,
    resume=True,                  # after a disconnect, just re-run this cell -> resumes
)
print(f"\nDONE in {(time.time() - t0) / 3600:.2f}h")
print(f"test NLL = {res['test_nll_mean']:.4f} +- {res['test_nll_se']:.4f}")

with open(os.path.join(OUTPUT, f"{DATASET}_bichannel_{GATE_MODE}_FINAL.pkl"), "wb") as f:
    pickle.dump(res, f)
print("saved FINAL pkl to Drive")

## If Colab disconnects

Per-fold and per-config checkpoints are stored on Drive under
`MyDrive/bichannel_cv/hard/`. Reconnect and re-run the **Run CV** cell --
`resume=True` continues from the last checkpoint (at most ~1 config is lost).

When all four runs finish, each leaves `suthana_bichannel_<mode>_FINAL.pkl`
in its Drive folder. Compare `test_nll_mean` across additive / learnable /
surprise / hard -- and against the earlier vanilla_rnn / rl_ann / context_ann /
memory_ann suthana numbers.